# Neural Network Classification

Predicting a category instead of a number. Two problems, one workflow:

- **Binary classification** - two classes (red or blue dots from `make_circles`).
- **Multiclass classification** - ten classes (Fashion-MNIST clothing images).

What we'll cover:

- The typical architecture of a classification network.
- Input and output shapes, and the losses that match them.
- Building a training loop from scratch.
- Why non-linearity is required for non-linear data.
- Decision boundaries, train/test splits and loss curves.
- Finding a good learning rate.
- Evaluation beyond accuracy: confusion matrix and classification report.
- Why normalizing inputs matters.
- Peeking at the weights a model learns.

> Runs locally with `uv run jupyter lab` and in Colab. The first cell installs
> anything Colab is missing.

## Typical architecture of a classification network

| Hyperparameter | Binary classification | Multiclass classification |
| --- | --- | --- |
| Input shape | one value per feature (2 here) | one value per feature (784 here) |
| Hidden layers | at least 1, problem specific | same |
| Hidden activation | usually ReLU | same |
| Output shape | 1 value (probability of class 1) | 1 value per class (10 here) |
| Output activation | sigmoid (inside the loss) | softmax (inside the loss) |
| Loss | `nn.BCEWithLogitsLoss` | `nn.CrossEntropyLoss` |
| Optimizer | SGD or Adam | same |

Two PyTorch details worth remembering:

- `nn.BCEWithLogitsLoss` expects **raw logits** and applies the sigmoid internally.
- `nn.CrossEntropyLoss` expects **raw logits** and integer class labels; it applies softmax internally.

So the model's final layer has no activation. This is a common source of confusion.

In [ ]:
# Colab setup: install notebook-only extras that Colab does not ship with.
import sys

if "google.colab" in sys.modules:
    try:
        import torchinfo  # noqa: F401
    except ModuleNotFoundError:
        import subprocess

        subprocess.run(
            [sys.executable, "-m", "pip", "-q", "install", "torchinfo"],
            check=True,
        )

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_circles
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Pick the best available device (CUDA / Apple MPS / CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


def repo_root(start=None):
    """Walk upwards until we find the folder that holds pyproject.toml."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


DATA_DIR = repo_root() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch {torch.__version__} | device: {device}")
print(f"Data folder: {DATA_DIR}")

**What this cell does**
- Installs `torchinfo` in Colab if it is missing, then loads the libraries we need.
- Seeds Python, NumPy and PyTorch so runs are repeatable.
- Picks CUDA, Apple MPS or CPU, and points `DATA_DIR` at the shared `data/` folder.

## Creating data to view and fit

Start with a toy dataset before touching a real one. `make_circles` gives two
interleaved circles: class `0` in the middle and class `1` around it.

> A model that can only draw straight lines cannot separate circles. Keep that
> in mind for the next few cells.

In [ ]:
# Make 1,000 examples: two features and one binary label
X, y = make_circles(n_samples=1000, noise=0.03, random_state=42)

# Convert to float32 tensors; models expect float inputs
X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(dim=1).to(device)

print("X:", X.shape, "| y:", y.shape)

In [ ]:
# Look at the features and labels as a table
circles = pd.DataFrame({
    "X0": X[:, 0].cpu().numpy(),
    "X1": X[:, 1].cpu().numpy(),
    "label": y.squeeze().cpu().numpy().astype(int),
})

print(circles.head())
print("\nClass counts:")
print(circles["label"].value_counts())

In [ ]:
# Visualize the two rings
plt.scatter(X[:, 0].cpu(), X[:, 1].cpu(), c=y.squeeze().cpu(), cmap=plt.cm.RdYlBu)
plt.title("Two classes, shaped as circles")
plt.xlabel("X0")
plt.ylabel("X1")
plt.show()

**What this shows**
- 1,000 rows, 2 features, and a balanced set of two labels.
- The classes form rings, not two groups that a single straight line can split.

Visualizing the data first is not optional. It tells you what kind of model the
problem actually needs.

## Input and output shapes

Most confusing PyTorch errors start with a shape mismatch. Check the shapes
before building anything.

- **Input shape:** 2 values per example -> `nn.Linear(in_features=2, ...)`.
- **Output shape:** 1 value per example -> `nn.Linear(..., out_features=1)`.

In [ ]:
print("X shape:", X.shape)          # (1000, 2) = samples, features
print("y shape:", y.shape)          # (1000, 1) = samples, target
print("Rows:", len(X), "| labels:", len(y))
print("First example:", X[0], "| label:", y[0])

`y` has a trailing dimension of 1 because `BCEWithLogitsLoss` compares a
`(N, 1)` prediction with a `(N, 1)` target. For multiclass later, the target is
`(N,)` because `CrossEntropyLoss` takes one integer label per example.

## Steps in modelling

Every PyTorch model follows the same pattern:

1. **Create** the model with `nn.Sequential` (or a custom `nn.Module`).
2. **Choose a loss** that matches the task and an **optimizer** that updates the weights.
3. **Train** with an explicit loop:
   forward pass -> loss -> `zero_grad()` -> `backward()` -> `step()`.

There is no `model.fit()` in PyTorch. The loop is yours, which makes every step visible.

In [ ]:
# 1. Create the simplest possible model: 2 inputs -> 1 output
torch.manual_seed(42)
model_1 = nn.Sequential(nn.Linear(in_features=2, out_features=1)).to(device)

# 2. Binary loss + optimizer
loss_fn = nn.BCEWithLogitsLoss()   # sigmoid + cross-entropy in one stable step
optimizer = optim.SGD(model_1.parameters(), lr=0.01)

# 3. Full-batch training loop
epochs = 200
for epoch in range(epochs):
    model_1.train()

    logits = model_1(X)            # raw scores, not probabilities
    loss = loss_fn(logits, y)

    optimizer.zero_grad()          # clear old gradients
    loss.backward()                # compute new gradients
    optimizer.step()               # nudge the weights

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | loss: {loss.item():.4f}")

# 4. Accuracy: sigmoid > 0.5 is the same as logit > 0
model_1.eval()
with torch.inference_mode():
    logits = model_1(X)
    train_acc = ((logits > 0).float() == y).float().mean()

print(f"Training accuracy: {train_acc.item() * 100:.1f}%")

**What this shows**
- A single `Linear(2, 1)` reaches about **50% accuracy** - the same as guessing.
- It is not broken. It can only draw one straight line, and a line cannot
  separate two rings.

A 50% score on balanced binary data is a useful warning sign: the model has not
learned the pattern.

## Visualize the decision boundary

Accuracy is one number. A decision boundary shows **where** the model puts the
cut-off.

The helper below:

1. Samples a grid of points across the feature space.
2. Predicts a class for every grid point.
3. Colours the regions and overlays the real data.

In [ ]:
def plot_decision_boundary(model, X, y):
    """Plot the regions a model predicts, with the real points on top."""
    model.eval()
    X_cpu = X.detach().cpu()
    y_cpu = y.detach().cpu()

    # Bounds of the plot, with a small margin
    x_min, x_max = X_cpu[:, 0].min() - 0.1, X_cpu[:, 0].max() + 0.1
    y_min, y_max = X_cpu[:, 1].min() - 0.1, X_cpu[:, 1].max() + 0.1

    # A meshgrid is every combination of x and y we want to predict on
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 100),
        np.linspace(y_min, y_max, 100),
    )
    grid = torch.tensor(
        np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32
    ).to(device)

    with torch.inference_mode():
        logits = model(grid)

    # One output = binary (threshold at 0); many outputs = multiclass (argmax)
    if logits.shape[1] == 1:
        preds = (logits > 0).float().cpu().numpy().reshape(xx.shape)
        print("binary decision boundary")
    else:
        preds = logits.argmax(dim=1).cpu().numpy().reshape(xx.shape)
        print("multiclass decision boundary")

    plt.contourf(xx, yy, preds, cmap=plt.cm.RdYlBu, alpha=0.7)
    plt.scatter(X_cpu[:, 0], X_cpu[:, 1], c=y_cpu.squeeze(), s=25,
                cmap=plt.cm.RdYlBu, edgecolors="k")
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())


plot_decision_boundary(model_1, X, y)
plt.title("A linear model can only draw a straight line")
plt.show()

The boundary is straight. More layers alone will not fix this if they stay linear.

## Stacking linear layers is still linear

Two `nn.Linear` layers with no activation between them are mathematically just
one linear layer. The network has more weights, but it still only draws a line.

Let's prove it with a deeper model.

In [ ]:
# Reusable helpers for the rest of the notebook
def batch_accuracy(logits, y):
    """Accuracy for binary (1 output) or multiclass (many outputs) logits."""
    if logits.shape[1] == 1:
        return ((logits > 0).float() == y).sum().item()
    return (logits.argmax(dim=1) == y).sum().item()


def evaluate_model(model, loader, loss_fn):
    """Return average loss and accuracy over a DataLoader."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.inference_mode():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)

            running_loss += loss_fn(logits, y_batch).item() * X_batch.size(0)
            correct += batch_accuracy(logits, y_batch)
            total += X_batch.size(0)

    return running_loss / total, correct / total


def train_model(model, train_loader, loss_fn, optimizer, epochs=100, test_loader=None):
    """Train a model and record loss/accuracy for plotting."""
    history = {"loss": [], "accuracy": []}
    if test_loader is not None:
        history["test_loss"] = []
        history["test_accuracy"] = []

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # The five training steps, in order
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
            correct += batch_accuracy(logits, y_batch)
            total += X_batch.size(0)

        history["loss"].append(running_loss / total)
        history["accuracy"].append(correct / total)

        if test_loader is not None:
            test_loss, test_acc = evaluate_model(model, test_loader, loss_fn)
            history["test_loss"].append(test_loss)
            history["test_accuracy"].append(test_acc)

    return history

In [ ]:
# A deep stack, but every layer is still linear
torch.manual_seed(42)
model_2 = nn.Sequential(
    nn.Linear(2, 8),
    nn.Linear(8, 8),
    nn.Linear(8, 1),
).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_2.parameters(), lr=0.01)
binary_loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

history_2 = train_model(model_2, binary_loader, loss_fn, optimizer, epochs=100)

plot_decision_boundary(model_2, X, y)
plt.title("Still a straight line - no activation was used")
plt.show()

print(f"Accuracy: {history_2['accuracy'][-1] * 100:.1f}%")

Still about 50%. Composition of linear functions is another linear function, so
depth without activation buys nothing.

> This is the key idea: **linear + non-linear functions are what let a network
> draw complex patterns.**

## Adding non-linearity

Activation functions bend the line:

- **ReLU** turns every negative value into `0` and leaves positives unchanged.
- **Sigmoid** squashes any value into `(0, 1)`.
- **Linear** changes nothing at all.

Let's apply each one to a toy tensor.

In [ ]:
A = torch.arange(-10, 10, dtype=torch.float32)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(A, torch.sigmoid(A))
axes[0].set_title("Sigmoid")

axes[1].plot(A, torch.relu(A))
axes[1].set_title("ReLU")

axes[2].plot(A, A)
axes[2].set_title("Linear (no change)")

for ax in axes:
    ax.grid(True)
plt.show()

`torch.sigmoid` and `torch.relu` bend the input. The linear function returns the
exact same values, which is why a purely linear network cannot learn a curved
boundary.

In a classification network, ReLU goes between hidden layers. The output layer
stays linear because the loss function applies sigmoid (binary) or softmax
(multiclass) for us.

## Evaluating on unseen data

Everything so far was scored on the same data the model trained on. That can
look good while telling you nothing about generalization.

Split the data:

- **Training set (80%)** - the model learns from this.
- **Test set (20%)** - stays untouched until the final check.

With 1,000 examples that is 800 for training and 200 for testing.

In [ ]:
X_train, y_train = X[:800], y[:800]
X_test, y_test = X[800:], y[800:]

print(f"Train: {len(X_train)} examples")
print(f"Test:  {len(X_test)} examples")

`X_train` and `y_train` are used for the weights. `X_test` and `y_test` are only
used for evaluation.

Now the same shape of network, but this time with ReLU between the layers.

In [ ]:
torch.manual_seed(42)

# 2 inputs -> 8 hidden -> 8 hidden -> 1 output
model_3 = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),                 # the missing piece from model_2
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 1),           # raw logit; BCEWithLogitsLoss applies sigmoid
).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_3.parameters(), lr=0.001)   # Adam's default

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

history_3 = train_model(
    model_3,
    train_loader,
    loss_fn,
    optimizer,
    epochs=100,
    test_loader=test_loader,
)

test_loss, test_acc = evaluate_model(model_3, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc * 100:.1f}%")

In [ ]:
# Compare the boundaries on training and test data
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_3, X_train, y_train)

plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_3, X_test, y_test)

plt.show()

The boundary now wraps around the rings, and test accuracy is close to 100%.

The network did not memorize a list of points; it learned a curved rule that
generalizes to unseen examples.

In [ ]:
# Plot loss and accuracy for both train and test
pd.DataFrame(history_3).plot(figsize=(9, 5))
plt.title("Training curves")
plt.xlabel("Epoch")
plt.ylabel("Loss / accuracy")
plt.grid(True)
plt.show()

**Reading the curves**
- Loss goes down and accuracy goes up - the model is improving.
- Test curves follow the train curves, so there is no obvious overfitting here.

If the train loss kept falling while the test loss turned upward, that would be
overfitting: memorizing the training set instead of learning the pattern.

## Finding a good learning rate

The learning rate controls how big each weight update is:

- Too small -> training crawls.
- Too large -> the loss jumps around or explodes.
- About right -> the loss falls quickly and settles.

A **learning-rate sweep** finds a sensible starting point. Start tiny, multiply
the rate a little every epoch, and record the loss. The useful range is where the
loss is still falling, before the curve turns upward.

In [ ]:
torch.manual_seed(42)

# A fresh model, so the sweep starts from a clean state
model_lr = nn.Sequential(
    nn.Linear(2, 8), nn.ReLU(),
    nn.Linear(8, 8), nn.ReLU(),
    nn.Linear(8, 1),
).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_lr.parameters(), lr=1e-4)
lr_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

lrs, losses = [], []

for epoch in range(100):
    # 1e-4 -> about 10 over 100 epochs
    lr = 1e-4 * 10 ** (epoch / 20)
    for group in optimizer.param_groups:
        group["lr"] = lr

    model_lr.train()
    running_loss = 0.0

    for X_batch, y_batch in lr_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model_lr(X_batch)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    lrs.append(lr)
    losses.append(running_loss / len(X_train))

# Log scale on x, because learning rates span orders of magnitude
plt.figure(figsize=(9, 5))
plt.semilogx(lrs, losses)
plt.xlabel("Learning rate (log scale)")
plt.ylabel("Loss")
plt.title("Learning rate vs. loss")
plt.grid(True)
plt.show()

Pick a rate where the loss is still falling, before the jumpy or exploding region.
Here that is around **0.01**.

The exact bottom is not a safe target: the curve is already noisy there, and a
small change in the data can push training over the edge.

In [ ]:
# Refit with the learning rate we picked
torch.manual_seed(42)

model_4 = nn.Sequential(
    nn.Linear(2, 8), nn.ReLU(),
    nn.Linear(8, 8), nn.ReLU(),
    nn.Linear(8, 1),
).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_4.parameters(), lr=0.01)

history_4 = train_model(
    model_4,
    train_loader,
    loss_fn,
    optimizer,
    epochs=20,
    test_loader=test_loader,
)

test_loss, test_acc = evaluate_model(model_4, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc * 100:.1f}%")

With `lr=0.01` the model reaches a similar accuracy in **20 epochs** instead of
100. Choosing a learning rate is an experiment: start with a sensible value,
measure, then adjust.

> Adam's default `lr=0.001` is a good first try for most problems. A sweep like
> the one above is still worth running when training feels slow.

## More evaluation metrics

Accuracy is one number. These metrics answer different questions:

| Metric | Question it answers |
| --- | --- |
| Accuracy | Out of all predictions, how many were correct? |
| Precision | When the model says "class 1", how often is it right? |
| Recall | Of all the real class 1 examples, how many did it find? |
| F1-score | A single balance of precision and recall. |
| Confusion matrix | Which classes are being confused with which? |
| Classification report | Precision, recall and F1 for every class at once. |

Accuracy can look great while a model ignores a rare class. Always look at the
confusion matrix for classification problems.

In [ ]:
# Get probabilities and hard labels for the test set
model_4.eval()
with torch.inference_mode():
    test_logits = model_4(X_test)
    test_probs = torch.sigmoid(test_logits).squeeze().cpu().numpy()

# 0.5 is the usual threshold. Change it to trade precision against recall.
test_preds = (test_probs > 0.5).astype(int)
y_test_np = y_test.squeeze().cpu().numpy().astype(int)

print("First 10 probabilities:", np.round(test_probs[:10], 3))
print("First 10 predictions:  ", test_preds[:10])

print(f"\nAccuracy:  {accuracy_score(y_test_np, test_preds):.3f}")
print(f"Precision: {precision_score(y_test_np, test_preds):.3f}")
print(f"Recall:    {recall_score(y_test_np, test_preds):.3f}")
print(f"F1-score:  {f1_score(y_test_np, test_preds):.3f}")

A model outputs **probabilities**, not labels. `sigmoid(logit)` gives the
probability of class 1, and comparing it with `0.5` turns it into a label.

The threshold is a choice. Raising it makes the model more conservative about
predicting class 1; lowering it does the opposite.

In [ ]:
import itertools


def make_confusion_matrix(y_true, y_pred, classes=None, figsize=(8, 8), text_size=12):
    """Plot a labelled confusion matrix with counts and row percentages."""
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    n_classes = cm.shape[0]

    fig, ax = plt.subplots(figsize=figsize)
    cax = ax.matshow(cm, cmap=plt.cm.Blues)
    fig.colorbar(cax)

    labels = classes if classes is not None else np.arange(n_classes)

    ax.set(
        title="Confusion Matrix",
        xlabel="Predicted label",
        ylabel="True label",
        xticks=np.arange(n_classes),
        yticks=np.arange(n_classes),
        xticklabels=labels,
        yticklabels=labels,
    )
    ax.xaxis.set_label_position("bottom")
    ax.xaxis.tick_bottom()

    threshold = (cm.max() + cm.min()) / 2.0

    for i, j in itertools.product(range(n_classes), range(n_classes)):
        ax.text(
            j, i,
            f"{cm[i, j]} ({cm_norm[i, j] * 100:.1f}%)",
            ha="center", va="center", size=text_size,
            color="white" if cm[i, j] > threshold else "black",
        )

    plt.show()


make_confusion_matrix(y_test_np, test_preds, classes=["Class 0", "Class 1"])

Correct predictions sit on the diagonal (top-left to bottom-right). Everything
off the diagonal is a mistake.

For more than two classes, this same function shows exactly **which** pairs the
model mixes up - much more useful than a single accuracy number.

## Multiclass classification: Fashion-MNIST

Binary classification is one thing or another. Multiclass classification picks
one of many options.

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) gives:

- 60,000 training images and 10,000 test images.
- 28x28 grayscale images.
- 10 clothing classes.

`torchvision` downloads the dataset into the shared `data/` folder on first run.

In [ ]:
from torchvision import datasets

train_set = datasets.FashionMNIST(root=str(DATA_DIR), train=True, download=True)
test_set = datasets.FashionMNIST(root=str(DATA_DIR), train=False, download=True)

print(train_set)
print("Train images:", train_set.data.shape)
print("Test images: ", test_set.data.shape)

In [ ]:
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

print("Number of classes:", len(class_names))
print("First training label:", train_set.targets[0].item())
print("Its class name:      ", class_names[train_set.targets[0]])

In [ ]:
# Plot a few random training images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for ax in axes.flatten():
    idx = random.randrange(len(train_set))
    ax.imshow(train_set.data[idx], cmap="binary")
    ax.set_title(f"{class_names[train_set.targets[idx]]}\nlabel={train_set.targets[idx].item()}")
    ax.axis(False)

plt.tight_layout()
plt.show()

**What this shows**
- Each input is a 28x28 image; each label is an integer from `0` to `9`.
- The images are pixel values from `0` to `255`, not probabilities.
- The shapes of clothing are not straight lines, so hidden ReLU layers are needed.

For `nn.Linear`, each image must be flattened from `28x28` to `784` values.

## Flatten and normalize

Two preprocessing steps:

1. **Flatten** - `(N, 28, 28)` -> `(N, 784)` so a `Linear` layer can accept it.
2. **Normalize** - divide pixel values by `255` so inputs are in `[0, 1]`.

Normalization keeps activations and gradients in a healthy range, which usually
means faster and more stable training.

In [ ]:
# Flatten images into one vector per example
X_train_flat = train_set.data.float().reshape(-1, 28 * 28)
X_test_flat = test_set.data.float().reshape(-1, 28 * 28)

# Labels stay as integers; CrossEntropyLoss expects class indices
y_train_mc = train_set.targets.long()
y_test_mc = test_set.targets.long()

# Normalized copies in [0, 1]
X_train_norm = X_train_flat / 255.0
X_test_norm = X_test_flat / 255.0

print("Flattened:", X_train_flat.shape, X_test_flat.shape)
print("Raw range:       ", X_train_flat.min().item(), "to", X_train_flat.max().item())
print("Normalized range:", X_train_norm.min().item(), "to", X_train_norm.max().item())

# Mini-batches of 64; shuffle only the training data
train_loader_raw = DataLoader(
    TensorDataset(X_train_flat, y_train_mc), batch_size=64, shuffle=True
)
test_loader_raw = DataLoader(
    TensorDataset(X_test_flat, y_test_mc), batch_size=64
)
train_loader_norm = DataLoader(
    TensorDataset(X_train_norm, y_train_mc), batch_size=64, shuffle=True
)
test_loader_norm = DataLoader(
    TensorDataset(X_test_norm, y_test_mc), batch_size=64
)

## Same model, different data: normalization matters

We will train the **exact same architecture** twice:

- Once on raw pixels (`0-255`).
- Once on normalized pixels (`0-1`).

Only the data changes. This is a controlled experiment.

In [ ]:
def build_mlp(hidden_units=4):
    """A small multilayer perceptron for 28x28 images and 10 classes."""
    return nn.Sequential(
        nn.Linear(784, hidden_units),
        nn.ReLU(),
        nn.Linear(hidden_units, hidden_units),
        nn.ReLU(),
        nn.Linear(hidden_units, 10),   # raw logits; CrossEntropyLoss applies softmax
    ).to(device)


loss_fn = nn.CrossEntropyLoss()

# --- Raw pixels (0-255) ---
torch.manual_seed(42)
model_raw = build_mlp(hidden_units=4)
optimizer_raw = optim.Adam(model_raw.parameters(), lr=0.001)

non_norm_history = train_model(
    model_raw, train_loader_raw, loss_fn, optimizer_raw, epochs=10
)
raw_loss, raw_acc = evaluate_model(model_raw, test_loader_raw, loss_fn)

print(f"Raw pixels   -> test loss: {raw_loss:.3f} | test accuracy: {raw_acc * 100:.1f}%")

In [ ]:
# --- Normalized pixels (0-1), same architecture ---
torch.manual_seed(42)
model_norm = build_mlp(hidden_units=4)
optimizer_norm = optim.Adam(model_norm.parameters(), lr=0.001)

norm_history = train_model(
    model_norm, train_loader_norm, loss_fn, optimizer_norm, epochs=10
)
norm_loss, norm_acc = evaluate_model(model_norm, test_loader_norm, loss_fn)

print(f"Normalized   -> test loss: {norm_loss:.3f} | test accuracy: {norm_acc * 100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(non_norm_history["loss"], label="raw pixels (0-255)")
axes[0].plot(norm_history["loss"], label="normalized (0-1)")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(non_norm_history["accuracy"], label="raw pixels (0-255)")
axes[1].plot(norm_history["accuracy"], label="normalized (0-1)")
axes[1].set_title("Training accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

**The lesson:** the architecture did not change, but the result did.

- Raw pixels: the model barely learns (~10% - guessing among 10 classes).
- Normalized pixels: the same model reaches roughly 70%.

The same model with even slightly different data can produce dramatically
different results. When comparing models, change **one thing at a time**.

## Finding the learning rate for multiclass

The same sweep works here. We use a 10,000-example subset to keep it quick - the
shape of the curve is what matters.

In [ ]:
torch.manual_seed(42)

model_lr_mc = build_mlp(hidden_units=4)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_lr_mc.parameters(), lr=1e-5)

# A subset is enough to see where the loss starts to explode
X_lr = X_train_norm[:10000]
y_lr = y_train_mc[:10000]
lr_loader_mc = DataLoader(TensorDataset(X_lr, y_lr), batch_size=64, shuffle=True)

lrs, losses = [], []

for epoch in range(30):
    # 1e-5 -> about 6 over 30 epochs
    lr = 1e-5 * 10 ** (epoch / 5)
    for group in optimizer.param_groups:
        group["lr"] = lr

    model_lr_mc.train()
    running_loss = 0.0

    for X_batch, y_batch in lr_loader_mc:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model_lr_mc(X_batch)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    lrs.append(lr)
    losses.append(running_loss / len(X_lr))

plt.figure(figsize=(9, 5))
plt.semilogx(lrs, losses)
plt.xlabel("Learning rate (log scale)")
plt.ylabel("Loss")
plt.title("Learning rate vs. loss (Fashion-MNIST)")
plt.grid(True)
plt.show()

The loss falls steadily from `1e-5` to about `4e-3`, then turns sharply upward.
A safe choice is `0.001`: still on the falling side, and it is Adam's default.

A sweep is a guide, not a final answer. It changes the rate every epoch, so the
exact minimum is not necessarily the best rate for a long run.

Now build a slightly wider model (`32` hidden units instead of `4`) and train it
for 20 epochs. More capacity means it can learn richer pixel patterns.

In [ ]:
torch.manual_seed(42)

final_model = build_mlp(hidden_units=32)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(final_model.parameters(), lr=0.001)

final_history = train_model(
    final_model,
    train_loader_norm,
    loss_fn,
    optimizer,
    epochs=20,
    test_loader=test_loader_norm,
)

final_loss, final_acc = evaluate_model(final_model, test_loader_norm, loss_fn)
print(f"Final test loss: {final_loss:.3f}")
print(f"Final test accuracy: {final_acc * 100:.1f}%")

In [ ]:
pd.DataFrame(final_history).plot(figsize=(9, 5))
plt.title("Fashion-MNIST training curves")
plt.xlabel("Epoch")
plt.ylabel("Loss / accuracy")
plt.grid(True)
plt.show()

About **87%** test accuracy from a tiny network. The remaining errors are mostly
visually similar classes, which the confusion matrix will show.

> To push higher you would add capacity, regularization or (much better for
> images) convolutional layers. The CNN notebook in this repository does that.

## Evaluating multiclass predictions

`CrossEntropyLoss` gives raw logits. To turn them into probabilities, apply
`softmax` across the 10 classes. The class with the highest probability is the
prediction.

In [ ]:
final_model.eval()
with torch.inference_mode():
    test_logits_mc = final_model(X_test_norm.to(device))
    test_probs_mc = torch.softmax(test_logits_mc, dim=1).cpu()

# Each row is one image; the 10 values add up to 1
print("Probability row sums:", test_probs_mc.sum(dim=1)[:5])

y_pred_mc = test_probs_mc.argmax(dim=1).numpy()
y_true_mc = y_test_mc.numpy()

print("\nFirst 5 predicted labels:", y_pred_mc[:5])
print("First 5 true labels:     ", y_true_mc[:5])
print("First 5 class names:     ", [class_names[i] for i in y_pred_mc[:5]])

In [ ]:
make_confusion_matrix(
    y_true=y_true_mc,
    y_pred=y_pred_mc,
    classes=class_names,
    figsize=(12, 12),
    text_size=9,
)

In [ ]:
print(classification_report(
    y_true_mc,
    y_pred_mc,
    target_names=class_names,
))

**Reading the results**
- The diagonal is bright: most classes are predicted correctly.
- The most common mistakes are between similar-looking classes, especially
  `Shirt` vs `T-shirt/top` and `Sneaker` vs `Ankle boot`.
- Those pairs share an overall shape, which is exactly the kind of pattern a
  small fully-connected network struggles to separate.

The classification report shows per-class precision and recall. A low recall for
one class means the model misses many examples of it, even if overall accuracy
looks fine.

In [ ]:
def plot_random_predictions(model, images, true_labels, classes):
    """Plot random images with predicted and true labels."""
    model.eval()
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))

    for ax in axes.flatten():
        idx = random.randrange(len(images))
        image = images[idx]

        with torch.inference_mode():
            logits = model(image.reshape(1, -1).to(device))
            probs = torch.softmax(logits, dim=1)

        pred_idx = probs.argmax(dim=1).item()
        confidence = probs.max().item() * 100
        true_idx = true_labels[idx].item()

        ax.imshow(image.reshape(28, 28), cmap="binary")
        ax.set_title(
            f"Pred: {classes[pred_idx]} ({confidence:.0f}%)\nTrue: {classes[true_idx]}",
            color="green" if pred_idx == true_idx else "red",
            fontsize=10,
        )
        ax.axis(False)

    plt.tight_layout()
    plt.show()


plot_random_predictions(final_model, X_test_norm, y_test_mc, class_names)

Run that cell a few times. You will see that most mistakes are genuinely
ambiguous even to a human: a boot that looks like a sneaker, a shirt that looks
like a top.

Visual inspection is a sanity check that numbers alone cannot give you.

## What patterns did the model learn?

A `nn.Linear` layer stores a weight for every input-output pair. For the first
layer that is `784` weights per hidden unit - one weight per pixel.

Those weights form a 28x28 template showing which pixels the unit responds to.

In [ ]:
# First layer of the model: (hidden_units, 784)
first_layer = final_model[0]
print("Weight shape:", first_layer.weight.shape)

weights = first_layer.weight.detach().cpu()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(weights[i].reshape(28, 28), cmap="RdBu_r")
    ax.set_title(f"Hidden unit {i}")
    ax.axis(False)

plt.suptitle("First-layer weights (red = positive, blue = negative)")
plt.tight_layout()
plt.show()

In PyTorch a `Linear` weight matrix has shape `(out_features, in_features)`, so
each **row** is one neuron's template. Some libraries store the matrix the other
way round, so check the shape when comparing weight matrices.

Each value says how strongly a pixel pushes that neuron up or down. The weights
start random and are updated during training until they represent useful
patterns. These templates are not clean drawings of shirts or bags - the useful
work happens across many units together.

In [ ]:
summary(final_model, input_size=(1, 784))

**Counting parameters**

For `nn.Linear(in, out)` the parameter count is `in * out + out` (weights plus
one bias per output).

- `Linear(784, 32)` = 784 * 32 + 32 = 25,120
- `Linear(32, 32)` = 32 * 32 + 32 = 1,056
- `Linear(32, 10)` = 32 * 10 + 10 = 330

Total: **26,506** trainable parameters. Small models can still do useful work.

## How a model learns (in brief)

Each epoch repeats the same cycle:

1. **Forward pass** - data flows through the layers and produces predictions.
2. **Loss** - the predictions are compared with the true labels.
3. **Backward pass** - backpropagation computes how much each weight contributed
   to the error.
4. **Optimizer step** - each weight is nudged in the direction that lowers the loss.

Weights and biases are the model's **parameters**; they are learned. Layer
counts, learning rate and epochs are **hyperparameters**; you choose them.

The loss curve is your feedback: if it is not going down, something in the data,
model or learning rate needs attention.

## Exercises

1. Create a dataset with `make_moons(noise=0.1)`, visualize it, and train a model
   to at least 85% test accuracy.
2. Push Fashion-MNIST past 88% by changing hidden units, epochs or learning rate.
   Plot the confusion matrix afterwards and see which classes improved.
3. Implement softmax yourself:
   `exp(x) / exp(x).sum(dim=1, keepdim=True)`. Compare it with `torch.softmax`.
4. Plot three images of the same class (for example `T-shirt/top`) with their
   predictions. Which one is the model least confident about?
5. Remove the ReLU layers from the binary model and retrain. Explain the result
   using the decision boundary.
6. Change the binary classification threshold from `0.5` to `0.8`. What happens
   to precision and recall?

## Extra resources

- [PyTorch tutorials](https://pytorch.org/tutorials/) - official, task-focused guides.
- [3Blue1Brown: But what is a neural network?](https://www.youtube.com/watch?v=aircAruvnKk)
  and [Gradient descent, how neural networks learn](https://www.youtube.com/watch?v=IHZwWFHWa-w)
  - visual intuition for forward and backward passes.
- [CS231n: Neural Networks Part 1](https://cs231n.github.io/neural-networks-1/) -
  activation functions, architecture and capacity.
- [Karpathy: A Recipe for Training Neural Networks](http://karpathy.github.io/2019/04/25/recipe/) -
  a practical debugging checklist.
- [Fashion-MNIST labels](https://github.com/zalandoresearch/fashion-mnist#labels) -
  the class names used here.